我们可以在工具或中间件中访问长期记忆。

## 1、在工具中访问长期记忆

### 1.1 基于InMemoryStore

In [1]:
import os

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv(override=True)

model = init_chat_model(
    model="deepseek-flash",
    model_provider="deepseek",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL")
)

In [4]:
from langchain_core.messages import HumanMessage
from langgraph.prebuilt import ToolRuntime
from langchain_core.tools import tool
from typing import NotRequired
from langchain.agents import AgentState, create_agent
from langgraph.store.memory import InMemoryStore

store = InMemoryStore()

class CustomState(AgentState):
    """继承AgentState，增加user_id字段"""
    user_id: NotRequired[str]

@tool(parse_docstring=True)
def save_user_info(name: str, runtime: ToolRuntime) -> str:
    """
    将用户信息保存在长期记忆中

    Args:
        name: 用户名

    Returns:
        str: 保存状态
    """
    runtime.store.put(("users",), runtime.state["user_id"], {"name": name})
    return "saved"

@tool(parse_docstring=True)
def get_user_info(runtime: ToolRuntime) -> str:
    """
    从长期记忆中读取用户信息

    Returns:
        str: 用户信息
    """
    item = runtime.store.get(("users",), runtime.state["user_id"])
    return str(item.value) if item else "unknown"

agent = create_agent(
    model=model,
    tools=[save_user_info, get_user_info],
    store=store,
    system_prompt="用户提及个人信息时及时记录，用户询问个人信息时尝试用工具检索",
    state_schema=CustomState
)

print("=" * 30, '-> 第一个会话（线程） <-', "=" * 30)
response1 = agent.invoke({
    "messages": [HumanMessage("你好，很高兴认识你，我是小花")],
    "user_id": "user-1"
})
for msg in response1["messages"]:
    msg.pretty_print()

print("=" * 30, '-> 第二个会话（线程） <-', "=" * 30)
response2 = agent.invoke({
    "messages": [HumanMessage("我是谁")],
    "user_id": "user-1"
})
for msg in response2["messages"]:
    msg.pretty_print()

============================== -> 第一个会话（线程） <- ==============================
================================ Human Message =================================

你好，很高兴认识你，我是小花
================================== Ai Message ==================================

很高兴认识你，小花！我会记住你的名字。你的名字真好听呀 😊
Tool Calls:
  save_user_info (call_00_ynUogvWkUZYv9vsnENWx5257)
 Call ID: call_00_ynUogvWkUZYv9vsnENWx5257
  Args:
    name: 小花
================================= Tool Message =================================
Name: save_user_info

saved
================================== Ai Message ==================================

已经记住啦，小花！以后我就能用你的名字来称呼你了 🌸

有什么我可以帮你的吗？无论是聊天、解答问题，还是别的什么，我都很乐意帮忙！
============================== -> 第二个会话（线程） <- ==============================
================================ Human Message =================================

我是谁
================================== Ai Message ==================================
Tool Calls:
  get_user_info (call_00_ipX1yGgQTmxT0p3CdA6f5257)
 Call ID: call_00_ip

其中，CustomState扩展了 Agent 的标准状态。除了默认的 messages （历史消息列表）之外，还额外增加了一个 user_id 字段。这样，Agent 在运行过程中随时随地都能知道当前和它说话的用户 ID 是什么。

两次invoke没有通过config串联为起来，是两个独立的会话，但第二个会话可以访问第一个会话写入长期记忆的内容。

### 1.2 基于PostgresStore

In [5]:
from langgraph.store.postgres import PostgresStore

DB_URL="postgresql://" + os.getenv("POSTGRESQL_USER") + ":" + os.getenv("POSTGRESQL_PASSWORD") + "@localhost:5432/" + os.getenv("POSTGRESQL_DB") + "?sslmode=disable"

class CustomState(AgentState):
    """继承AgentState，增加user_id字段"""
    user_id: NotRequired[str]

@tool(parse_docstring=True)
def save_user_info(name: str, runtime: ToolRuntime) -> str:
    """
    将用户信息保存在长期记忆中

    Args:
        name: 用户名

    Returns:
        str: 保存状态
    """
    runtime.store.put(("users",), runtime.state["user_id"], {"name": name})
    return "saved"

@tool(parse_docstring=True)
def get_user_info(runtime: ToolRuntime) -> str:
    """
    从长期记忆中读取用户信息

    Returns:
        str: 用户信息
    """
    item = runtime.store.get(("users",), runtime.state["user_id"])
    return str(item.value) if item else "unknown"

with PostgresStore.from_conn_string(DB_URL) as store:
    store.setup()
    agent = create_agent(
        model=model,
        tools=[save_user_info, get_user_info],
        store=store,
        system_prompt="用户提及个人信息时及时记录，用户询问个人信息时尝试用工具检索",
        state_schema=CustomState
    )

    print("=" * 30, '-> 第一个会话（线程） <-', "=" * 30)
    response1 = agent.invoke({
        "messages": [HumanMessage("你好，很高兴认识你，我是小花")],
        "user_id": "user-1"
    })
    for msg in response1["messages"]:
        msg.pretty_print()

    print("=" * 30, '-> 第二个会话（线程） <-', "=" * 30)
    response2 = agent.invoke({
        "messages": [HumanMessage("我是谁")],
        "user_id": "user-1"
    })
    for msg in response2["messages"]:
        msg.pretty_print()

============================== -> 第一个会话（线程） <- ==============================
================================ Human Message =================================

你好，很高兴认识你，我是小花
================================== Ai Message ==================================

你好，小花！很高兴认识你 😊 我先把你的名字记下来。
Tool Calls:
  save_user_info (call_00_O3SX5YU0DJbpIAqzb4qA3070)
 Call ID: call_00_O3SX5YU0DJbpIAqzb4qA3070
  Args:
    name: 小花
================================= Tool Message =================================
Name: save_user_info

saved
================================== Ai Message ==================================

已经记住啦，小花！以后我们聊天我都能认出你来。有什么想聊的或者需要帮忙的，尽管告诉我～
============================== -> 第二个会话（线程） <- ==============================
================================ Human Message =================================

我是谁
================================== Ai Message ==================================
Tool Calls:
  get_user_info (call_00_m0PJxnyeojSZMBTqtZYH7148)
 Call ID: call_00_m0PJxnyeojSZMBTqtZYH7148
  

## 2、在中间件中访问长期记忆

### 2.1 Node-style hooks中访问

以before_model为例，其钩子函数签名如下
```python
def before_model(self, state: StateT, runtime: Runtime[ContextT]) -> dict[str, Any] | None:
    """Logic to run before the model is called.

    Args:
        state: The current agent state.
        runtime: The runtime context.

    Returns:
        Agent state updates to apply before model call.
    """
```
Runtime定义如下
```python
@dataclass(**_DC_KWARGS)
class Runtime(Generic[ContextT]):
    context: ContextT = field(default=None)  # type: ignore[assignment]
    """Static context for the graph run, like `user_id`, `db_conn`, etc.

    Can also be thought of as 'run dependencies'."""

    store: BaseStore | None = field(default=None)
    """Store for the graph run, enabling persistence and memory."""

    stream_writer: StreamWriter = field(default=_no_op_stream_writer)
    """Function that writes to the custom stream."""

    heartbeat: Callable[[], None] = field(default=_no_op_heartbeat)
    """Record progress for the current node's `idle_timeout`.

    Call this from inside long-running work that does not naturally emit
    writes, stream chunks, child tasks, or LangChain callback events, to
    prevent the node from being treated as idle. It is also the only
    progress signal honored under `TimeoutPolicy(refresh_on="heartbeat")`.
    Outside an idle-timed attempt this is a no-op.
    """

    previous: Any = field(default=None)
    """The previous return value for the given thread.

    Only available with the functional API when a checkpointer is provided.
    """

    execution_info: ExecutionInfo | None = field(default=None)
    """Read-only execution information/metadata for the current node run.

    None before task preparation populates it."""

    server_info: ServerInfo | None = field(default=None)
    """Metadata injected by LangGraph Server. None when running open-source LangGraph without LangSmith deployments."""

    control: RunControl | None = field(default=None)
    """Run-scoped control plane for cooperative draining.

    Populated automatically during graph runs. None outside an active
    graph runtime.
    """
```
所以，我们可以通过 runtime.store 在中间件中访问长期记忆。

### 2.2 Wrap-style hooks中访问

1、wrap_model_call

钩子函数签名如下
```python
def wrap_model_call(
        self,
        request: ModelRequest[ContextT],
        handler: Callable[[ModelRequest[ContextT]], ModelResponse[ResponseT]],
    ) -> ModelResponse[ResponseT] | AIMessage | ExtendedModelResponse[ResponseT]:
```
ModelRequest定义如下：
```python
@dataclass(init=False)
class ModelRequest(Generic[ContextT]):
    """Model request information for the agent.

    Type Parameters:
        ContextT: The type of the runtime context.

            Defaults to `None` if not specified.
    """

    model: BaseChatModel
    messages: list[AnyMessage]  # excluding system message
    system_message: SystemMessage | None
    tool_choice: Any | None
    tools: list[BaseTool | dict[str, Any]]
    response_format: ResponseFormat[Any] | None
    state: AgentState[Any]
    runtime: Runtime[ContextT]
    model_settings: dict[str, Any] = field(default_factory=dict)
    # ...
```
所以，可以通过 request.runtime.store 访问长期记忆。

2、wrap_tool_call

钩子函数签名如下
```python
def wrap_tool_call(
        self,
        request: ToolCallRequest,
        handler: Callable[[ToolCallRequest], ToolMessage | Command[Any]],
    ) -> ToolMessage | Command[Any]:
```
ToolCallRequest定义如下：
```python
@dataclass
class ToolCallRequest:
    """Tool execution request passed to tool call interceptors.

    Attributes:
        tool_call: Tool call dict with name, args, and id from model output.
        tool: BaseTool instance to be invoked, or None if tool is not
            registered with the `ToolNode`. When tool is `None`, interceptors can
            handle the request without validation. If the interceptor calls `execute()`,
            validation will occur and raise an error for unregistered tools.
        state: Agent state (`dict`, `list`, or `BaseModel`).
        runtime: LangGraph runtime context (optional, `None` if outside graph).
    """

    tool_call: ToolCall
    tool: BaseTool | None
    state: Any
    runtime: ToolRuntime
    # ...
```
ToolRuntime定义如下：
```python
@dataclass
class ToolRuntime(_DirectlyInjectedToolArg, Generic[ContextT, StateT]):
    state: StateT
    context: ContextT
    config: RunnableConfig
    stream_writer: StreamWriter
    tool_call_id: str | None
    store: BaseStore | None
```
所以，可以通过 request.runtime.store 访问长期记忆。